`python
# ================================================================
# NOTEBOOK 1: CHUẨN BỊ 200 CÂU TEST CHO THỰC NGHIỆM E5 - TTS
# Môi trường: Google Colab | Runtime: CPU là đủ
# ================================================================
`

In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 1 — Mount Google Drive (CHẠY ĐẦU TIÊN)              │
# └─────────────────────────────────────────────────────────────┘
# Colab sẽ hỏi xin quyền → Nhấn "Connect to Google Drive"
# rồi đăng nhập tài khoản Google của bạn

from google.colab import drive
drive.mount('/content/drive')

# ---- Tạo thư mục lưu trữ trên Drive ----
import os
DRIVE_DIR = '/content/drive/MyDrive/E5_TTS_Benchmark'
os.makedirs(DRIVE_DIR, exist_ok=True)

print("✅ Đã kết nối Google Drive thành công!")
print(f"📁 Thư mục lưu trữ: {DRIVE_DIR}")
print()
print("👉 Kiểm tra thư mục này trên Drive của bạn tại:")
print("   https://drive.google.com  →  Thư mục 'E5_TTS_Benchmark'")

In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 2 — Cài đặt thư viện                                │
# └─────────────────────────────────────────────────────────────┘

!pip install datasets pandas tqdm --quiet

print("✅ Đã cài đặt xong các thư viện.")

In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 3 — Import và cấu hình chung                        │
# └─────────────────────────────────────────────────────────────┘

import json
import pandas as pd
import random
from pathlib import Path
from datasets import load_dataset
from tqdm import tqdm

random.seed(42)  # Cố định seed để tái lập được

# Dùng thư mục Drive làm đầu ra chính (không lưu vào /content/ tạm thời)
OUTPUT_DIR = Path(DRIVE_DIR)

print(f"✅ Cấu hình xong — Đầu ra sẽ lưu vào Drive: {OUTPUT_DIR}")

In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 4 — Tải và lọc 100 câu từ UIT-VSFC (ĐÃ SỬA LỖI)     │
# └─────────────────────────────────────────────────────────────┘
# Dataset: UIT-VSFC (dùng mirror Parquet chuẩn, tương thích datasets mới nhất)

print("⏳ Đang tải UIT-VSFC từ HuggingFace...")

vsfc_dataset = None

# Thử tải từ bản Parquet mirror chuẩn (không bị lỗi script .py)
try:
    vsfc_dataset = load_dataset("hkab/vietnamese_students_feedback", split="train")
    print(f"✅ Tải thành công từ 'hkab/vietnamese_students_feedback': {len(vsfc_dataset)} mẫu")
except Exception as e:
    print(f"   Thử split 'test'...")
    try:
        vsfc_dataset = load_dataset("hkab/vietnamese_students_feedback", split="test")
        print(f"✅ Tải thành công split 'test': {len(vsfc_dataset)} mẫu")
    except Exception as e2:
        print("   Thử tải trực tiếp file parquet...")
        parquet_url = "hf://datasets/uitnlp/vietnamese_students_feedback@refs/convert/parquet/default/train/0000.parquet"
        vsfc_dataset = load_dataset("parquet", data_files=parquet_url, split="train")

# In tên các cột để kiểm tra
print(f"   Các cột trong dataset: {vsfc_dataset.column_names}")

# ---- Lọc câu theo tiêu chí E5 ----
MIN_CHARS = 40    # Tối thiểu 40 ký tự (đủ để đo TTFA có ý nghĩa)
MAX_CHARS = 300   # Tối đa 300 ký tự (độ dài 1 đoạn trả lời ngắn)

vsfc_sentences = []
seen = set()

# Tự động phát hiện tên cột text
sample_keys = list(vsfc_dataset[0].keys())
text_col = "sentence" if "sentence" in sample_keys else sample_keys[0]
print(f"   Dùng cột text: '{text_col}'")

for item in tqdm(vsfc_dataset, desc="Lọc câu UIT-VSFC"):
    text = str(item.get(text_col, "")).strip()
    if not text or text in seen:
        continue
    if not (MIN_CHARS <= len(text) <= MAX_CHARS):
        continue
    if any(c in text for c in ["<", ">", "{", "}", "\\", "http"]):
        continue
    seen.add(text)
    vsfc_sentences.append(text)

random.shuffle(vsfc_sentences)
vsfc_100 = vsfc_sentences[:100]

print(f"\n✅ Đã lọc xong — chọn {len(vsfc_100)} câu từ UIT-VSFC")
print(f"   Ví dụ 2 câu đầu:")
for i, s in enumerate(vsfc_100[:2], 1):
    print(f"   [{i}] {s}")

In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 5 — Tạo 100 câu HUIT Placeholder                    │
# └─────────────────────────────────────────────────────────────┘
# ⚠️ ĐÂY LÀ DỮ LIỆU MẪU GIẢ ĐỊNH để chạy thử pipeline.
#    Thay bằng gold_answer thật từ HUIT-RAGBench khi nhóm hoàn thành.

HUIT_PLACEHOLDER_SENTENCES = [
    # === Nhóm 1: Học phí và tài chính (15 câu) ===
    "Học phí học kỳ 1 năm học 2024-2025 được thu theo tín chỉ, với mức 550.000 đồng mỗi tín chỉ đối với sinh viên hệ đại trà.",
    "Sinh viên có thể nộp học phí qua cổng thanh toán trực tuyến tại địa chỉ portal.huit.edu.vn hoặc nộp trực tiếp tại phòng Kế hoạch - Tài chính trong giờ hành chính.",
    "Mức học phí cho chương trình đào tạo tiên tiến và chất lượng cao được áp dụng riêng, cao hơn so với hệ đại trà theo quy định của Bộ Giáo dục và Đào tạo.",
    "Sinh viên được giảm 50% học phí khi có giấy chứng nhận hộ nghèo hoặc cận nghèo hợp lệ do UBND xã, phường, thị trấn cấp.",
    "Học phí phải được đóng đầy đủ trước ngày 30 của tháng đầu tiên trong học kỳ để không bị khóa tài khoản đăng ký học phần.",
    "Trường hợp sinh viên rút khỏi học phần trong vòng 2 tuần đầu tiên của học kỳ, học phí đã nộp sẽ được hoàn trả 70% sau khi trừ phí xử lý hồ sơ.",
    "Sinh viên hệ vừa học vừa làm đóng học phí theo mức riêng, được quy định trong thông báo đầu mỗi học kỳ từ Phòng Đào tạo.",
    "Để được miễn giảm học phí, sinh viên cần nộp hồ sơ trước ngày 15 tháng 9 đối với học kỳ 1 và trước ngày 15 tháng 2 đối với học kỳ 2.",
    "Học phí chương trình liên kết quốc tế với các trường đối tác được tính theo thỏa thuận riêng và có thể thanh toán một phần bằng ngoại tệ.",
    "Trường không hoàn trả học phí cho sinh viên tự ý bỏ học giữa chừng trừ trường hợp có lý do bất khả kháng được Hội đồng xét duyệt chấp thuận.",
    "Học bổng khuyến khích học tập được xét mỗi học kỳ dựa trên điểm trung bình tích lũy, với mức từ 1 đến 3 lần học phí tiêu chuẩn.",
    "Phòng Kế hoạch - Tài chính làm việc từ thứ Hai đến thứ Sáu, từ 7 giờ 30 đến 11 giờ 30 và từ 13 giờ 30 đến 17 giờ.",
    "Sinh viên nhận học bổng ngoài trường cần thông báo cho Phòng Công tác Sinh viên trong vòng 30 ngày kể từ ngày nhận để cập nhật hồ sơ.",
    "Mức học phí được công bố công khai và cập nhật hàng năm trên trang thông tin điện tử chính thức của trường tại địa chỉ www.huit.edu.vn.",
    "Sinh viên có thể đăng ký vay vốn tín dụng sinh viên tại Ngân hàng Chính sách Xã hội với lãi suất ưu đãi theo chính sách của Chính phủ.",
    # === Nhóm 2: Đăng ký học phần (15 câu) ===
    "Thời gian đăng ký học phần học kỳ 1 năm học 2024-2025 bắt đầu từ ngày 10 tháng 6 và kết thúc vào ngày 30 tháng 6 năm 2024.",
    "Sinh viên đăng ký học phần theo đúng lịch phân công trên hệ thống, ưu tiên theo thứ tự từ sinh viên năm cuối đến sinh viên năm nhất.",
    "Mỗi học kỳ chính, sinh viên được đăng ký tối thiểu 12 tín chỉ và tối đa 25 tín chỉ tùy theo điểm trung bình tích lũy của học kỳ trước.",
    "Sinh viên muốn đăng ký quá tải trên 22 tín chỉ cần điểm trung bình tích lũy từ 3.2 trở lên và được cố vấn học tập chấp thuận.",
    "Hệ thống đăng ký học phần mở từ 7 giờ sáng, sinh viên cần chuẩn bị danh sách học phần muốn đăng ký trước để tránh bị đầy lớp.",
    "Sau khi kết thúc đợt đăng ký chính thức, sinh viên chỉ được điều chỉnh học phần trong tuần đầu tiên của học kỳ qua phòng Đào tạo.",
    "Học phần có điều kiện tiên quyết chỉ được đăng ký khi sinh viên đã hoàn thành và đạt điểm D trở lên ở học phần tiên quyết đó.",
    "Trường hợp học phần không đủ số lượng sinh viên tối thiểu, lớp học có thể bị hủy và sinh viên được ưu tiên chuyển sang lớp khác.",
    "Sinh viên có thể tra cứu lịch học, phòng học và giảng viên phụ trách qua hệ thống quản lý đào tạo tại địa chỉ đms.huit.edu.vn.",
    "Cố vấn học tập được phân công cho từng lớp và chịu trách nhiệm hỗ trợ sinh viên trong việc lập kế hoạch học tập phù hợp.",
    "Sinh viên nghỉ học tạm thời được bảo lưu kết quả học tập và tiếp tục đăng ký học phần khi quay lại theo đúng quy định.",
    "Học phần tự chọn cần được đăng ký trong danh sách đã được phê duyệt theo chương trình đào tạo của từng ngành học cụ thể.",
    "Sinh viên không đủ điều kiện đăng ký học phần do nợ học phí sẽ bị hệ thống từ chối tự động cho đến khi hoàn thành nghĩa vụ tài chính.",
    "Lịch thi cuối kỳ được công bố ít nhất 2 tuần trước ngày thi, sinh viên cần kiểm tra và phản hồi ngay nếu có trùng lịch.",
    "Điểm danh bằng ứng dụng điện thoại hoặc thẻ sinh viên tùy thuộc vào quy định của từng giảng viên phụ trách học phần.",
    # === Nhóm 3: Tuyển sinh (10 câu) ===
    "Trường Đại học Công Thương Thành phố Hồ Chí Minh tuyển sinh dựa trên nhiều phương thức, bao gồm xét điểm thi THPT, xét học bạ và xét kết quả thi đánh giá năng lực.",
    "Điểm chuẩn tuyển sinh năm 2024 dao động từ 18 đến 26 điểm tùy theo ngành, được công bố chính thức trên trang tuyển sinh của trường.",
    "Thí sinh xét tuyển học bạ cần điểm trung bình cả năm lớp 12 từ 6.5 trở lên và đáp ứng điều kiện riêng của từng ngành.",
    "Ngành Công nghệ thông tin và Kỹ thuật phần mềm thuộc nhóm có điểm chuẩn cao nhất trường, phù hợp với thí sinh có định hướng công nghệ.",
    "Hồ sơ nhập học cần chuẩn bị đầy đủ bao gồm bằng tốt nghiệp THPT hoặc bằng tương đương, học bạ gốc và ảnh thẻ theo quy chuẩn.",
    "Trường có chính sách ưu tiên xét tuyển cho thí sinh là con em gia đình chính sách, dân tộc thiểu số và thí sinh ở vùng khó khăn.",
    "Chương trình liên kết đào tạo quốc tế tuyển sinh riêng và yêu cầu thí sinh có năng lực ngoại ngữ, thường là chứng chỉ IELTS từ 5.5 trở lên.",
    "Thông tin tuyển sinh chi tiết theo từng ngành, chỉ tiêu và học phí được công bố trên trang tuyển sinh tại địa chỉ tuyensinh.huit.edu.vn.",
    "Sinh viên trúng tuyển cần hoàn tất thủ tục nhập học theo đúng lịch thông báo, bao gồm đóng học phí và kích hoạt tài khoản hệ thống.",
    "Trường tổ chức ngày hội tư vấn tuyển sinh hàng năm, cho phép thí sinh tham quan cơ sở vật chất và gặp gỡ trực tiếp với cố vấn tuyển sinh.",
    # === Nhóm 4: Tốt nghiệp và bằng cấp (10 câu) ===
    "Để đăng ký xét tốt nghiệp, sinh viên cần hoàn thành toàn bộ khối lượng học tập theo chương trình đào tạo, đạt chuẩn đầu ra ngoại ngữ và tin học.",
    "Chuẩn đầu ra ngoại ngữ bắt buộc từ khóa 2021 là chứng chỉ tiếng Anh tương đương IELTS 4.5 hoặc TOEIC 450 điểm trở lên.",
    "Sinh viên nợ môn phải hoàn thành tất cả các học phần bắt buộc trước khi nộp hồ sơ xét tốt nghiệp, bao gồm cả các học phần thực tập.",
    "Bằng tốt nghiệp được cấp theo loại: xuất sắc, giỏi, khá, trung bình khá, trung bình dựa trên điểm trung bình tích lũy toàn khóa học.",
    "Thời gian nhận bằng tốt nghiệp thường từ 3 đến 6 tháng sau lễ tốt nghiệp, sinh viên có thể đến Phòng Đào tạo để nhận theo lịch thông báo.",
    "Sinh viên cần nộp bản xác nhận hoàn thành nghĩa vụ tài chính và không vi phạm kỷ luật mới được xem xét xét tốt nghiệp.",
    "Luận văn tốt nghiệp phải đạt điểm từ 5.0 trở lên và được Hội đồng chấm luận văn thông qua theo quy trình do Phòng Đào tạo quy định.",
    "Sinh viên có thể đăng ký cấp bảng điểm, xác nhận sinh viên và giấy chứng nhận tốt nghiệp tạm thời qua cổng dịch vụ trực tuyến của trường.",
    "Điều kiện xét tốt nghiệp loại giỏi yêu cầu điểm trung bình tích lũy từ 3.2 đến 3.59 và không có học phần nào bị điểm F trong toàn khóa.",
    "Sinh viên hoàn thành chương trình sớm hơn kế hoạch có thể nộp đơn xét tốt nghiệp sớm và sẽ được xét trong đợt gần nhất tiếp theo.",
    # === Nhóm 5: Kỷ luật và quy chế (10 câu) ===
    "Sinh viên vắng mặt quá 20% số tiết học của một học phần sẽ bị cấm thi học phần đó và nhận điểm F cho học kỳ tương ứng.",
    "Hành vi gian lận trong thi cử, bao gồm quay cóp, mang tài liệu trái phép và sử dụng thiết bị điện tử, bị xử lý nghiêm theo quy định.",
    "Sinh viên có thể khiếu nại kết quả thi trong vòng 5 ngày làm việc kể từ ngày công bố điểm bằng cách nộp đơn tại Phòng Khảo thí.",
    "Vi phạm quy chế lần đầu có thể bị cảnh cáo hoặc đình chỉ học tập từ 1 đến 3 tháng tùy mức độ vi phạm.",
    "Sinh viên bị buộc thôi học do vi phạm kỷ luật có quyền khiếu nại lên Hội đồng Kỷ luật trường trong vòng 15 ngày.",
    "Bảo vệ tài sản công và giữ gìn vệ sinh môi trường trường học là nghĩa vụ của tất cả sinh viên theo quy định của nhà trường.",
    "Sinh viên sử dụng đồng phục theo quy định trong các buổi học lý thuyết và thực hành, đặc biệt trong những tuần đầu tiên của học kỳ.",
    "Hoạt động ngoại khóa, câu lạc bộ và tổ chức sự kiện phải được Phòng Công tác Sinh viên phê duyệt trước ít nhất 3 ngày làm việc.",
    "Sinh viên mượn thiết bị phòng thí nghiệm phải ký mượn và hoàn trả đúng hạn, chịu trách nhiệm bồi thường nếu làm hỏng hoặc mất.",
    "Giải quyết khiếu nại và thắc mắc về học tập là trách nhiệm của cố vấn học tập, Phòng Đào tạo và Ban Giám hiệu theo từng cấp độ.",
    # === Nhóm 6: Dịch vụ sinh viên (10 câu) ===
    "Thư viện trường mở cửa từ thứ Hai đến thứ Sáu từ 7 giờ đến 20 giờ và thứ Bảy từ 7 giờ đến 17 giờ để phục vụ nhu cầu học tập.",
    "Sinh viên có thể mượn tối đa 5 cuốn sách với thời hạn mượn là 14 ngày và có thể gia hạn thêm một lần nếu không có người đặt trước.",
    "Phòng máy tính tại tầng 3 nhà A phục vụ sinh viên tra cứu tài liệu và học trực tuyến, cần xuất trình thẻ sinh viên khi vào.",
    "Xe đạp và xe máy sinh viên phải gửi đúng bãi quy định, nhà trường không chịu trách nhiệm với xe gửi sai khu vực.",
    "Nhà ký túc xá của trường tiếp nhận sinh viên đến từ các tỉnh thành khác, ưu tiên sinh viên có hoàn cảnh khó khăn và điểm học tập tốt.",
    "Phòng Y tế trường cung cấp dịch vụ khám sức khỏe ban đầu và sơ cứu, đặt tại tầng 1 nhà Hành chính, phục vụ trong giờ hành chính.",
    "Canteen trường được vận hành bởi đơn vị được kiểm duyệt vệ sinh an toàn thực phẩm, phục vụ từ 6 giờ 30 đến 19 giờ 30 các ngày trong tuần.",
    "Phòng Công tác Sinh viên tiếp nhận đơn xin trợ cấp khó khăn đột xuất, sinh viên cần nộp kèm minh chứng theo hướng dẫn trong biểu mẫu.",
    "Câu lạc bộ học thuật, văn nghệ và thể thao hoạt động dưới sự quản lý của Đoàn Thanh niên và Hội Sinh viên trường.",
    "Mạng WiFi nội bộ của trường phủ sóng toàn bộ khuôn viên, sinh viên đăng nhập bằng tài khoản email trường cấp để sử dụng.",
    # === Nhóm 7: Chương trình và ngành học (10 câu) ===
    "Trường đào tạo hơn 40 ngành học ở trình độ đại học, sau đại học bao gồm kỹ thuật, kinh tế, công nghệ thông tin, quản trị và ngôn ngữ.",
    "Ngành Công nghệ thực phẩm là một trong những ngành thế mạnh của trường với đội ngũ giảng viên giàu kinh nghiệm và phòng thí nghiệm hiện đại.",
    "Chương trình đào tạo được cập nhật định kỳ theo nhu cầu thị trường lao động và tham khảo ý kiến từ doanh nghiệp đối tác của trường.",
    "Sinh viên ngành Kế toán được học phần mềm kế toán thực hành Misa và phần mềm thuế trực tiếp tại phòng máy tính của khoa.",
    "Chương trình song ngữ Anh-Việt trong một số ngành giúp sinh viên nâng cao năng lực ngoại ngữ đáp ứng yêu cầu doanh nghiệp FDI.",
    "Thực tập tốt nghiệp thường kéo dài từ 10 đến 15 tuần tại doanh nghiệp đối tác, được tính điểm như học phần bắt buộc trong chương trình.",
    "Ngành Quản trị kinh doanh có chuyên ngành Quản trị du lịch và lữ hành, phù hợp với sinh viên muốn phát triển sự nghiệp trong ngành dịch vụ.",
    "Sinh viên hoàn thành chương trình kỹ sư 4 năm có thể liên thông lên bậc thạc sĩ với thời gian học rút ngắn theo thỏa thuận đào tạo.",
    "Chương trình giáo dục đại cương trong 2 học kỳ đầu giúp sinh viên xây dựng nền tảng kiến thức và kỹ năng cơ bản theo định hướng ngành.",
    "Danh sách học phần bắt buộc và tự chọn theo từng ngành được công bố chi tiết trong chuẩn chương trình đào tạo trên cổng thông tin trường.",
    # === Nhóm 8: Thực hành và kỹ năng mềm (10 câu) ===
    "Trung tâm Hỗ trợ Nghề nghiệp và Khởi nghiệp tổ chức các buổi hội thảo kỹ năng mềm và kết nối thực tập hàng tháng.",
    "Sinh viên có thể tham gia chương trình thực tập ngắn hạn tại các công ty đối tác ngay từ năm 2 theo chương trình internship của trường.",
    "Kỹ năng thuyết trình, làm việc nhóm và viết báo cáo kỹ thuật được lồng ghép vào nhiều học phần chuyên ngành từ năm thứ 2.",
    "Phòng thí nghiệm ngành Điện-Điện tử được trang bị thiết bị đo lường hiện đại, phục vụ thực hành cho sinh viên theo lịch phân công từng tuần.",
    "Các cuộc thi học thuật như Olympic Toán, Olympic Tiếng Anh và cuộc thi ý tưởng khởi nghiệp được tổ chức định kỳ hàng năm.",
    "Chương trình mentor kết nối sinh viên năm cuối với cựu sinh viên thành công trong ngành để trao đổi kinh nghiệm nghề nghiệp thực tế.",
    "Hệ thống học trực tuyến LMS của trường cung cấp bài giảng video, tài liệu và bài tập cho phép sinh viên học mọi lúc mọi nơi.",
    "Kết quả thi rèn luyện điểm cuối học kỳ ảnh hưởng trực tiếp đến việc xét học bổng và các danh hiệu sinh viên tiêu biểu.",
    "Giải thưởng sinh viên nghiên cứu khoa học cấp trường được trao hàng năm, khuyến khích sinh viên tham gia nghiên cứu từ sớm.",
    "Trung tâm Ngoại ngữ của trường cung cấp các khóa học tiếng Anh tập trung theo chuẩn IELTS và TOEIC với học phí ưu đãi cho sinh viên.",
]

assert len(HUIT_PLACEHOLDER_SENTENCES) == 100, \
    f"Cần đúng 100 câu, hiện có {len(HUIT_PLACEHOLDER_SENTENCES)}"

print(f"✅ Đã tạo {len(HUIT_PLACEHOLDER_SENTENCES)} câu HUIT placeholder")

In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 6 — Gộp và lưu benchmark_200.json lên Google Drive  │
# └─────────────────────────────────────────────────────────────┘

# Tự động lấy danh sách câu HUIT (ưu tiên câu thật từ CELL 5 mới nếu có)
if 'HUIT_REAL_SENTENCES' in globals():
    huit_sentences = HUIT_REAL_SENTENCES
    huit_source = globals().get('SOURCE_LABEL', 'HUIT-official')
    is_placeholder = globals().get('IS_PLACEHOLDER', False)
else:
    huit_sentences = HUIT_PLACEHOLDER_SENTENCES
    huit_source = "HUIT-holdout-placeholder"
    is_placeholder = True

def get_domain(idx):
    if idx < 15:   return "hoc_phi"
    elif idx < 30: return "dang_ky_hoc_phan"
    elif idx < 40: return "tuyen_sinh"
    elif idx < 50: return "tot_nghiep"
    elif idx < 60: return "ky_luat"
    elif idx < 70: return "dich_vu"
    elif idx < 80: return "chuong_trinh"
    else:          return "thuc_hanh"

benchmark_data = []

# 100 câu HUIT
for idx, text in enumerate(huit_sentences):
    benchmark_data.append({
        "question_id"  : f"HUIT_{idx+1:03d}",
        "source"       : huit_source,
        "text"         : text,
        "char_len"     : len(text),
        "word_count"   : len(text.split()),
        "is_placeholder": is_placeholder,
        "domain"       : get_domain(idx),
    })

# 100 câu UIT-VSFC
for idx, text in enumerate(vsfc_100):
    benchmark_data.append({
        "question_id"  : f"VSFC_{idx+1:03d}",
        "source"       : "UIT-VSFC",
        "text"         : text,
        "char_len"     : len(text),
        "word_count"   : len(text.split()),
        "is_placeholder": False,
        "domain"       : "student_feedback",
    })

# ---- Lưu lên Google Drive ----
json_path = OUTPUT_DIR / "benchmark_200.json"
csv_path  = OUTPUT_DIR / "benchmark_200.csv"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(benchmark_data, f, ensure_ascii=False, indent=2)

pd.DataFrame(benchmark_data).to_csv(csv_path, index=False, encoding="utf-8-sig")

print(f"✅ Đã lưu lên Google Drive:")
print(f"   📄 JSON : {json_path}")
print(f"   📊 CSV  : {csv_path}")
print(f"   📌 Nguồn HUIT: {huit_source} (is_placeholder={is_placeholder})")

In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 7 — Thống kê và xác nhận hoàn thành                 │
# └─────────────────────────────────────────────────────────────┘

df = pd.DataFrame(benchmark_data)

print("=" * 55)
print("📊 THỐNG KÊ BỘ DỮ LIỆU BENCHMARK E5")
print("=" * 55)
print(f"  Tổng số câu         : {len(df)}")
print(f"  Câu HUIT            : {len(df[df['source'].str.startswith('HUIT')])}")
print(f"  Câu UIT-VSFC (thật) : {len(df[df['source']=='UIT-VSFC'])}")
print()
print(f"  📏 Độ dài câu (ký tự):")
print(f"     Min   : {df['char_len'].min()}")
print(f"     Max   : {df['char_len'].max()}")
print(f"     Mean  : {df['char_len'].mean():.1f}")
print()
print("  📂 Phân bố theo domain:")
print(df.groupby('domain')['question_id'].count().to_string())
print("=" * 55)
print()
print("✅ NOTEBOOK 1 HOÀN THÀNH!")
print()
print("👉 BƯỚC TIẾP THEO:")
print(f"   1. Kiểm tra file trên Google Drive:")
print(f"      https://drive.google.com  →  Thư mục 'E5_TTS_Benchmark'")
print(f"   2. Mở Notebook 2 (runtime T4 GPU)")
print(f"   3. Chạy CELL 1 đầu tiên để mount Drive và tải file này về")